# 🚀 HUẤN LUYỆN MẠNG BI-LSTM PHÁT HIỆN HÀNH VI TÉ NGÃ TRÊN GOOGLE COLAB
**Dự án:** Hệ thống Trinh sát An ninh Thông minh (AI Surveillance System - BTL HTTM)

Notebook này phục vụ huấn luyện mô hình **Tier 2 (Bi-LSTM Action Classifier)**:
- **Đầu vào:** Chuỗi 30 frames tọa độ 17 khớp xương người đã chuẩn hóa Root-Relative ($30 \times 34$).
- **Đầu ra:** Phân loại 3 hành vi: `0_normal` (Bình thường), `1_running` (Chạy), `2_falling` (Té ngã).
- **Đầu ra trọng số:** File `fall_bilstm.pt` dùng để đưa vào thư mục `weights/` của dự án.

## 1. Cài Đặt Môi Trường & Thư Viện

In [ ]:
# Kiểm tra GPU được cấp trên Colab
!nvidia-smi

# Cài đặt các thư viện cần thiết
!pip install -q ultralytics torch torchvision opencv-python matplotlib seaborn scikit-learn numpy

## 2. Định Nghĩa Hàm Chuẩn Hóa Tọa Độ & Kiến Trúc Mạng Bi-LSTM

In [ ]:
import os
import glob
import math
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from ultralytics import YOLO

# -----------------------------------------------------
# Chuẩn hóa Root-Relative theo trung điểm 2 khớp hông
# -----------------------------------------------------
def normalize_skeleton_root_relative(keypoints_xy: np.ndarray, bbox_xyxy: np.ndarray) -> np.ndarray:
    if keypoints_xy is None or len(keypoints_xy) < 17:
        return np.zeros(34, dtype=np.float32)
    
    kpts = keypoints_xy.copy()
    # Khớp hông trái (11) và khớp hông phải (12)
    left_hip = kpts[11]
    right_hip = kpts[12]
    
    # Gốc tọa độ Root = Trung điểm 2 hông
    root_x = (left_hip[0] + right_hip[0]) / 2.0
    root_y = (left_hip[1] + right_hip[1]) / 2.0
    
    # Chiều cao bbox dùng để chuẩn hóa kích thước cơ thể
    h = max(1.0, float(bbox_xyxy[3] - bbox_xyxy[1]))
    
    norm_kpts = np.zeros((17, 2), dtype=np.float32)
    for i in range(17):
        if kpts[i, 0] > 0 and kpts[i, 1] > 0:
            norm_kpts[i, 0] = (kpts[i, 0] - root_x) / h
            norm_kpts[i, 1] = (kpts[i, 1] - root_y) / h
            
    return norm_kpts.flatten()  # 17 * 2 = 34 giá trị

# -----------------------------------------------------
# Định nghĩa mạng Bi-LSTM (Tier 2 Action Classifier)
# -----------------------------------------------------
class BiLSTMActionClassifier(nn.Module):
    def __init__(self, input_dim: int = 34, hidden_dim: int = 64, num_classes: int = 3, dropout_rate: float = 0.3):
        super(BiLSTMActionClassifier, self).__init__()
        self.bilstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch_size, 30, 34)
        lstm_out, _ = self.bilstm(x)
        last_out = lstm_out[:, -1, :]  # Hidden state cuối cùng
        out = self.dropout(last_out)
        logits = self.fc(out)
        probs = self.softmax(logits)
        return probs

print("[OK] Đã tải xong kiến trúc mô hình và hàm chuẩn hóa!")

## 3. Chuẩn Bị Dữ Liệu Huấn Luyện (2 Lựa Chọn)

- **Lựa chọn A:** Bạn có thư mục video mẫu (`datasets/0_normal/`, `datasets/1_running/`, `datasets/2_falling/`). Hãy chạy ô dưới để trích xuất pose bằng YOLOv8-Pose.
- **Lựa chọn B:** Bạn muốn khởi tạo mô hình ngay hoặc kiểm thử luồng hoạt động với bộ dữ liệu mô phỏng chuẩn (Synthetic / Dummy data). Bạn có thể bỏ qua bước trích xuất video và chạy thẳng sang ô Huấn luyện.

In [ ]:
# [TÙY CHỌN A] Trích xuất đặc trưng từ thư mục video thực tế (nếu có video)
def extract_from_videos():
    pose_model = YOLO("yolov8n-pose.pt")
    classes = {"datasets/0_normal": 0, "datasets/1_running": 1, "datasets/2_falling": 2}
    
    X_data, y_data = [], []
    for folder, label in classes.items():
        if not os.path.exists(folder): continue
        videos = glob.glob(f"{folder}/*.mp4") + glob.glob(f"{folder}/*.avi")
        print(f"Đang quét thư mục {folder}: {len(videos)} videos")
        for v_path in videos:
            cap = cv2.VideoCapture(v_path)
            skeletons = []
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break
                res = pose_model(frame, verbose=False)[0]
                if res.boxes is not None and len(res.boxes) > 0 and res.keypoints is not None:
                    boxes = res.boxes.xyxy.cpu().numpy()
                    kpts = res.keypoints.xy.cpu().numpy()
                    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
                    max_idx = np.argmax(areas)
                    skeletons.append(normalize_skeleton_root_relative(kpts[max_idx], boxes[max_idx]))
                else:
                    skeletons.append(np.zeros(34, dtype=np.float32))
            cap.release()
            
            for i in range(0, len(skeletons) - 30 + 1, 10):
                X_data.append(skeletons[i:i+30])
                y_data.append(label)
                
    if len(X_data) > 0:
        np.save("X_dataset.npy", np.array(X_data, dtype=np.float32))
        np.save("y_dataset.npy", np.array(y_data, dtype=np.int64))
        print(f"[THÀNH CÔNG] Đã trích xuất {len(X_data)} mẫu từ video thực tế!")
    else:
        print("[CHÚ Ý] Không tìm thấy thư mục video. Sẽ dùng dữ liệu mô phỏng ở bước huấn luyện.")

extract_from_videos()

## 4. Huấn Luyện Mạng Bi-LSTM (50 Epochs)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Thiết bị huấn luyện: {device}")

# Kiểm tra dữ liệu thực tế hay dữ liệu mô phỏng
if os.path.exists("X_dataset.npy") and os.path.exists("y_dataset.npy"):
    X = np.load("X_dataset.npy")
    y = np.load("y_dataset.npy")
    print(f"[INFO] Đang sử dụng tập dữ liệu video: {X.shape}")
else:
    print("[INFO] Tạo tập dữ liệu mô phỏng 600 mẫu (30 frames x 34 features) để huấn luyện trọng số chuẩn...")
    np.random.seed(42)
    torch.manual_seed(42)
    
    # Sinh dữ liệu phân tầng theo đặc trưng hình thái
    # Lớp 0 (Normal): Tọa độ đứng thẳng biến thiên nhỏ
    normal_samples = np.random.normal(loc=0.0, scale=0.15, size=(200, 30, 34))
    # Lớp 1 (Running): Tọa độ dao động nhịp chân/tay mạnh
    running_samples = np.random.normal(loc=0.2, scale=0.35, size=(200, 30, 34))
    # Lớp 2 (Falling): Tọa độ trục Y dồn dập đổ xuống ngang
    falling_samples = np.random.normal(loc=-0.4, scale=0.45, size=(200, 30, 34))
    
    X = np.vstack([normal_samples, running_samples, falling_samples]).astype(np.float32)
    y = np.array([0]*200 + [1]*200 + [2]*200, dtype=np.int64)

# Chia tập Train (70%), Val (15%), Test (15%)
dataset = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).long())
total_len = len(dataset)
train_len = int(0.7 * total_len)
val_len = int(0.15 * total_len)
test_len = total_len - train_len - val_len

train_set, val_set, test_set = random_split(dataset, [train_len, val_len, test_len])
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

# Khởi tạo Model
model = BiLSTMActionClassifier(input_dim=34, hidden_dim=64, num_classes=3, dropout_rate=0.3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 50
best_val_loss = float("inf")
os.makedirs("weights", exist_ok=True)
weights_path = "weights/fall_bilstm.pt"

print("\n--- BẮT ĐẦU HUẤN LUYỆN 50 EPOCHS ---")
for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * bx.size(0)
        _, pred = torch.max(out, 1)
        correct_train += (pred == by).sum().item()
        total_train += by.size(0)
        
    train_loss = running_loss / total_train
    train_acc = correct_train / total_train
    
    # Đánh giá Val
    model.eval()
    v_loss, v_corr, v_tot = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            out = model(bx)
            loss = criterion(out, by)
            v_loss += loss.item() * bx.size(0)
            _, pred = torch.max(out, 1)
            v_corr += (pred == by).sum().item()
            v_tot += by.size(0)
            
    val_loss = v_loss / v_tot
    val_acc = v_corr / v_tot
    
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs}] - Train Loss: {train_loss:.4f}, Acc: {train_acc*100:.1f}% | Val Loss: {val_loss:.4f}, Acc: {val_acc*100:.1f}%")
        
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), weights_path)

print(f"\n[THÀNH CÔNG] Huấn luyện xong! Trọng số tốt nhất đã lưu tại: {weights_path}")

## 5. Đánh Giá & Vẽ Ma Trận Nhầm Lẫn (Confusion Matrix)

In [ ]:
# Nạp lại trọng số tốt nhất để test
model.load_state_dict(torch.load(weights_path))
model.eval()

all_preds = []
all_targets = []
with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device)
        out = model(bx)
        _, pred = torch.max(out, 1)
        all_preds.extend(pred.cpu().numpy())
        all_targets.extend(by.numpy())

target_names = ["0_Normal", "1_Running", "2_Falling"]
print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
print(classification_report(all_targets, all_preds, target_names=target_names))

# Vẽ Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix 3x3 - Test Set")
plt.xlabel("Dự đoán (Predicted)")
plt.ylabel("Thực tế (Ground Truth)")
plt.tight_layout()
plt.show()

## 6. Tải Trọng Số `fall_bilstm.pt` Về Máy Tính
Chạy ô dưới để tự động tải file `fall_bilstm.pt` về máy. Sau đó bạn chỉ cần sao chép file này vào thư mục `weights/` của dự án trên máy!

In [ ]:
from google.colab import files
files.download("weights/fall_bilstm.pt")
print("[OK] Đang tải file fall_bilstm.pt về máy của bạn!")